# RefuelGuard-LM LoRA Fine-Tuning

Free-tier friendly workflow for Qwen2.5-0.5B-Instruct. The model explains simulator outputs; it does not replace rules or anomaly detection.

In [ ]:
!pip install -q torch transformers datasets peft accelerate pandas bitsandbytes

In [ ]:
from pathlib import Path
from datasets import load_dataset

DATA_DIR = Path('/content/orbital-refueling/llm/data')
TRAIN_FILE = DATA_DIR / 'train.jsonl'
VAL_FILE = DATA_DIR / 'val.jsonl'
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR = '/content/refuelguard-lm-lora'

dataset = load_dataset('json', data_files={'train': str(TRAIN_FILE), 'validation': str(VAL_FILE)})
dataset

In [ ]:
dataset['train'][0]

In [ ]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, DataCollatorForSeq2Seq, Trainer, TrainingArguments

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def format_prompt(instruction, user_input):
    return '<|system|>\nYou explain synthetic spacecraft refueling telemetry anomalies. Do not replace deterministic rules or ML detectors.\n<|user|>\n' + instruction + '\n\n' + user_input + '\n<|assistant|>\n'

def tokenize(example, max_length=768):
    prompt = format_prompt(example['instruction'], example['input'])
    answer = example['output'] + tokenizer.eos_token
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    answer_ids = tokenizer(answer, add_special_tokens=False)['input_ids']
    input_ids = (prompt_ids + answer_ids)[:max_length]
    labels = ([-100] * len(prompt_ids) + answer_ids)[:max_length]
    return {'input_ids': input_ids, 'attention_mask': [1] * len(input_ids), 'labels': labels}

tokenized = dataset.map(tokenize, remove_columns=dataset['train'].column_names)

In [ ]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    fp16=True,
    report_to='none',
    disable_tqdm=False,
)

trainer = Trainer(model=model, args=args, train_dataset=tokenized['train'], eval_dataset=tokenized['validation'], data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True))
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

In [ ]:
sample = dataset['validation'][0]
prompt = format_prompt(sample['instruction'], sample['input'])
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    ids = model.generate(**inputs, max_new_tokens=180, do_sample=False, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))